# Tổng quan Milestone 2: Phân tích Phân cấp, Chuẩn hóa & Khử trùng (25.000 Bài báo)

## 1. Thực thi toàn nhóm từ lạnh trên Google Colab CPU-only

Toàn bộ **25.000 bài báo / 41.498 phiên bản** của 5 thành viên được phân tích từ đầu (`publications_reused = 0`, cùng mã băm `code_hash = cf2de7a143c8...`) qua **5 phiên Google Colab Standard CPU-only** không gối giờ, sau đó hợp nhất và kiểm chuẩn toàn cục 5 bảng bằng [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
# Sơ đồ kiến trúc Pipeline M2 4 bước (sinh trực tiếp lúc biên dịch)
fig, ax = plt.subplots(figsize=(10.2, 2.25))
ax.set_xlim(0, 10.2); ax.set_ylim(0, 2.25); ax.axis("off")
stages_m2 = [
    ("Bước 1 • Gathering", "Dò root \\documentclass\nĐối sánh tiêu đề M1 (1.706 v)\nMở rộng \\input/\\include", "#0F4C81"),
    ("Bước 2 • AST Hierarchy", "Document → Section → List\n→ Sentence / Math / Figure\nBỏ References, giữ Appendix", "#0284C7"),
    ("Bước 3 • Chuẩn hóa & Bib", "Toán inline / equation chuẩn\nGộp .bib + \\bibitem + .bbl\nĐổi khóa \\cite{} xuyên ver", "#0D9488"),
    ("Bước 4 • Khử trùng & DAG", "SHA-256 kho elements (-35,0%)\nBibTeX union (-35,8%)\nSQLite DAG valid = True", "#059669"),
]
for idx, (st_title, st_desc, col) in enumerate(stages_m2):
    x0 = 0.15 + idx * 2.52
    ax.add_patch(FancyBboxPatch((x0, 0.22), 2.22, 1.72, boxstyle="round,pad=0.06,rounding_size=0.10",
                                facecolor="#F8FAFC", edgecolor=col, linewidth=1.5))
    ax.add_patch(FancyBboxPatch((x0, 1.48), 2.22, 0.46, boxstyle="round,pad=0.04,rounding_size=0.08",
                                facecolor=col, edgecolor=col, linewidth=1.0))
    ax.text(x0 + 1.11, 1.71, st_title, ha="center", va="center", color="white", fontsize=9.1, fontweight="bold")
    ax.text(x0 + 1.11, 0.84, st_desc, ha="center", va="center", color="#1E293B", fontsize=8.1, linespacing=1.35)
    if idx < 3:
        ax.add_patch(FancyArrowPatch((x0 + 2.28, 1.08), (x0 + 2.50, 1.08),
                                     arrowstyle="-|>,head_length=6,head_width=4", color="#64748B", lw=1.6))
ax.set_title("Hình M2.0. Chu trình phân tích phân cấp, chuẩn hóa và khử trùng 2 tầng xuyên phiên bản (Milestone 2)", pad=4)
plt.tight_layout(); plt.show()

In [ ]:
tex_total = m2["versions_with_tex"]
labels = [
    ("success", "Thành công hoàn toàn (success)", True),
    ("partial", "Trích xuất bán phần (partial — giữ cây hợp lệ)", True),
    ("empty_no_tex", "Không có TeX (PDF-only / no-LaTeX)", False),
    ("error", "Lỗi cú pháp TeX nặng / thiếu root (error)", True),
    ("error_missing_source", "Nguồn thượng nguồn M1 khuyết (169 HTTP 404 + 11 chờ)", False),
]
rows_st = []
for k, lbl, in_tex in labels:
    cnt = int(m2["status_counts"].get(k, 0))
    rows_st.append({
        "Trạng thái phân tích": lbl,
        "Số phiên bản": cnt,
        "% / Tổng 41.498 version": 100.0 * cnt / scope["versions"],
        "% / 38.554 version có TeX": (100.0 * cnt / tex_total) if in_tex else np.nan,
    })
st_df = pd.DataFrame(rows_st)
display(style_table(
    st_df,
    caption="Bảng M2.1. Phân bố trạng thái parser trên toàn bộ 41.498 phiên bản và trên 38.554 phiên bản có mã nguồn TeX (97,34% dựng được cây).",
    int_cols=["Số phiên bản"],
    pct_cols=["% / Tổng 41.498 version", "% / 38.554 version có TeX"],
    bar_cols={"Số phiên bản": ("rgba(5, 150, 105, 0.22)", None)},
))

pm = pd.DataFrame(m2["per_member"])
view_m2 = pd.DataFrame({
    "Thành viên": pm["member"],
    "Phiên bản": pm["versions"],
    "Có TeX": pm["tex_verified"],
    "Success": pm["success"],
    "Partial": pm["partial"],
    "Empty": pm["empty_no_tex"],
    "Lỗi": pm["error"],
    "Success / TeX (%)": 100.0 * pm["tex_success_rate"],
    "Parsed / TeX (%)": 100.0 * pm["tex_parsed_rate"],
    "Thời gian (s)": pm["runtime_sec"],
    "Peak RSS (MiB)": pm["peak_rss_mib"],
})
display(style_table(
    view_m2,
    caption="Bảng M2.2. Kết quả phân tích và hiệu năng Colab CPU-only đồng đều qua cả 5 phân vùng thành viên.",
    int_cols=["Phiên bản", "Có TeX", "Success", "Partial", "Empty", "Lỗi"],
    pct_cols=["Success / TeX (%)", "Parsed / TeX (%)"],
    bar_cols={"Parsed / TeX (%)": ("rgba(13, 148, 136, 0.22)", 100.0)},
))

dd_rows = []
for k, name_vi in [("elements", "Phần tử văn bản (elements)"), ("bibtex", "Mục tài liệu tham khảo (bibtex)")]:
    info = m2["deduplication"][k]
    dd_rows.append({
        "Kho dữ liệu khử trùng": name_vi,
        "Trước khử trùng": int(info["before"]),
        "Sau khử trùng (duy nhất)": int(info["after"]),
        "Bản sao loại bỏ": int(info["before"] - info["after"]),
        "Tỷ lệ giảm (%)": 100.0 * float(info["reduction_rate"]),
    })
display(style_table(
    pd.DataFrame(dd_rows),
    caption="Bảng M2.3. Hiệu quả khử trùng 2 tầng xuyên phiên bản trên 25.000 bài báo.",
    int_cols=["Trước khử trùng", "Sau khử trùng (duy nhất)", "Bản sao loại bỏ"],
    pct_cols=["Tỷ lệ giảm (%)"],
    bar_cols={"Tỷ lệ giảm (%)": ("var(--ah-bar-fill)", 50.0)},
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.1), gridspec_kw={"width_ratios": [1.18, 1.0]})
y1 = np.arange(len(st_df))
c1 = [BOOK_PALETTE["emerald"], BOOK_PALETTE["amber"], BOOK_PALETTE["slate"], BOOK_PALETTE["crimson"], "#94A3B8"]
ax1.barh(y1, st_df["Số phiên bản"], color=c1, height=0.58)
ax1.set_yticks(y1); ax1.set_yticklabels(["Success", "Partial", "Empty (no TeX)", "Lỗi parse", "Thiếu nguồn M1"])
ax1.invert_yaxis(); ax1.set_xlim(0, 42000); ax1.grid(axis="x")
ax1.set_xlabel("Số phiên bản"); ax1.set_title("Hình M2.1a. Trạng thái 41.498 phiên bản")
for idx_r, r in enumerate(st_df.to_dict("records")):
    ax1.text(r["Số phiên bản"] + 450, idx_r,
             f"{fmt_int(r['Số phiên bản'])} ({fmt_pct(r['% / Tổng 41.498 version'])})",
             va="center", fontsize=8.1, fontweight="bold")

# 100% Stacked Horizontal Bar đối sánh 5 thành viên
y2 = np.arange(len(pm))
tot_v = pm["versions"].values
s_p = 100.0 * pm["success"].values / tot_v
p_p = 100.0 * pm["partial"].values / tot_v
e_p = 100.0 * pm["empty_no_tex"].values / tot_v
err_p = 100.0 * (pm["error"].values + pm["error_missing_source"].values) / tot_v
ax2.barh(y2, s_p, color=BOOK_PALETTE["emerald"], height=0.56, label="Success")
ax2.barh(y2, p_p, left=s_p, color=BOOK_PALETTE["amber"], height=0.56, label="Partial")
ax2.barh(y2, e_p, left=s_p + p_p, color=BOOK_PALETTE["slate"], height=0.56, label="Empty")
ax2.barh(y2, err_p, left=s_p + p_p + e_p, color=BOOK_PALETTE["crimson"], height=0.56, label="Lỗi/404")
ax2.set_yticks(y2); ax2.set_yticklabels(pm["member"]); ax2.invert_yaxis()
ax2.set_xlim(0, 100); ax2.set_xlabel("Tỷ trọng phiên bản (%)")
ax2.set_title("Hình M2.1b. Độ đồng đều qua 5 thành viên")
ax2.legend(ncol=4, fontsize=7.6, loc="lower center", bbox_to_anchor=(0.5, -0.28), frameon=False)
for i_m, sp in enumerate(s_p):
    ax2.text(sp / 2, i_m, fmt_pct(sp, 1), ha="center", va="center", color="white", fontsize=8.0, fontweight="bold")
plt.tight_layout(); plt.show()

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [`src/parser/statistics.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/statistics.py).
- Mã nguồn triển khai: [`tools/m2_colab_runner.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).